## Usernames filter

This notebook aims to filter the raw usernames contained in `usernames.txt`, to keep only those created between March, 17th and September, 17th, and who let public their production.

### Chose subset of the total `usernames.txt`

In [ ]:
X = 2 # Or 2 or 3

In [ ]:
with open('usernames.txt', 'r', encoding='utf-8') as f:
    raw_names = f.readlines()

breakpoint = int(len(raw_names)/3)
batch = raw_names[breakpoint*(X-1):breakpoint*(X)]
if X == 3:
    batch += raw_names[breakpoint*(X)+1:]

with open(f'covid_users_{X}.txt', 'r', encoding='utf-8') as f:
    last_done = f.readlines()[-1]
    if "Last done:" in last_done:
        batch = batch[batch.index(last_done)+1 : ]
    else : 
        print('No past attempt saved.')

print(f'Remaining length batch: {len(batch)}')

### Scrap Reddit to filter each username in the batch

In [ ]:
import requests
from datetime import datetime
import time

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; scraper/1.0)"
}

start = datetime(2020, 3, 17)
end = datetime(2020, 9, 17)

to_save = []
missing = 0

for i, username in enumerate(batch):
    if i%20==0:
        print(f'Step {i}')
    time.sleep(1)
    try:
        r = requests.get(f"https://www.reddit.com/user/{username.strip('\n')}/about.json", headers=headers)
        r.raise_for_status()
        data = r.json()
        created_utc = data["data"]["created_utc"]
    except:
        # print(f'Username {username.strip('\n')} does not appear.')
        missing+=1
        if missing % 5 == 0: 
            print(f"Missing:{(missing*100)/i+1}%")
        continue
    date_regis = datetime.utcfromtimestamp(created_utc)
    if start <= date_regis <= end:
        r = requests.get(f"https://www.reddit.com/user/{username.strip('\n')}/.json", headers=headers)
        r.raise_for_status()
        data = r.json()
        if data['data']['children']:
            to_save.append(username)
            print(f'Found:{len(to_save)}, Among:{i+1}')

print(f'Missing:{missing}')

with open(f'covid_users_{X}.txt', 'w', encoding='utf-8') as f:
    for username in to_save:
        f.write(username+'\n')

In [ ]:
with open(f'covid_users_{X}.txt', 'a', encoding='utf-8') as f:
    for user in to_save:
        f.write(user)
    f.write(f'Last done: {username}')